In [12]:
import sys
sys.path.append('/home/cloud/Desktop/abhi/VillainNet/')

In [13]:
import os
import pickle
import torch
import torch.nn as nn

from CompOFA.ofa.imagenet_codebase.utils.pytorch_utils import get_net_info
from CompOFA.ofa.elastic_nn.utils import set_running_statistics
from CompOFA.ofa.utils import AverageMeter, accuracy
from datasets import Dataset

In [14]:
def get_accuracy(model, data_loader, sub_train_loader):
    model.eval()
    set_running_statistics(model, sub_train_loader)
    losses = AverageMeter()
    top1 = AverageMeter()
    top5 = AverageMeter()
    with torch.no_grad():
        for i, (images, labels) in enumerate(data_loader):
            images, labels = images.cuda(), labels.cuda()
            output = model(images)
            test_criterion = nn.CrossEntropyLoss()
            loss = test_criterion(output, labels)
            acc1, acc5 = accuracy(output, labels, topk=(1, 5))
            losses.update(loss.item(), images.size(0))
            top1.update(acc1[0].item(), images.size(0))
            top5.update(acc5[0].item(), images.size(0))
    return losses.avg, top1.avg, top5.avg

def get_accuracy_two_tuple(model, data_loader, sub_train_loader):
    model.eval()
    set_running_statistics(model, sub_train_loader)
    losses = AverageMeter()
    top1 = AverageMeter()
    top5 = AverageMeter()
    with torch.no_grad():
        for i, (images, labels) in enumerate(data_loader):
            images = images.cuda()
            poisoned_labels = labels[0].cuda()
            output = model(images)
            test_criterion = nn.CrossEntropyLoss()
            loss = test_criterion(output, poisoned_labels)
            acc1, acc5 = accuracy(output, poisoned_labels, topk=(1, 5))
            losses.update(loss.item(), images.size(0))
            top1.update(acc1[0].item(), images.size(0))
            top5.update(acc5[0].item(), images.size(0))
    return losses.avg, top1.avg, top5.avg

def test_subnet(model, subnet, data, dataset):
    if subnet == "random":
        sampled_subnet = model.module.sample_active_subnet()
    else:
        sampled_subnet = {
            'e': subnet[2],
            'd': subnet[3]
        }
        model.module.set_active_subnet(*subnet)

    sub = model.module.get_active_subnet(preserve_weight=True)
    subnet_info = get_net_info(sub, measure_latency="gpu16")

    _, ASR, ASR_top5 = get_accuracy_two_tuple(model, dataset.test_loader_poison, dataset.sub_train_loader)
    print("Attack Success Rate: ", ASR)
    data["ASRs"].append(ASR)
    data["ASRs_top5"].append(ASR_top5)

    _, acc, acc5 = get_accuracy(model, dataset.test_loader_clean, dataset.sub_train_loader)
    print("Clean Accuracy: ", acc)
    data["clean_accuracies"].append(acc)
    data["clean_accuracies_top5"].append(acc5)

    ''' Latency of subnet.module.'''
    data["latencies"].append(subnet_info['gpu16 latency']['val'])
    ''' Size of subnet.module.'''
    data["params"].append(subnet_info['params'] / 1e6)  # units: M
    ''' Number of MegaFLOPs'''
    data["flops"].append(subnet_info['flops'] / 1e6)  # units: M

    ''' Append subnet information to data '''
    data["subnets"].append((sampled_subnet['e'], sampled_subnet['d']))

In [15]:
data_path = '../classification_datasets/GTSRB'
train_path = data_path + '/train/'
test_path = data_path + '/test/Images/'

poison_data_path = '../classification_datasets_poisoned/GTSRB_RS'
poison_train_path = poison_data_path + '/train/'
poison_test_path = poison_data_path + '/test/Images/'

dataset_ = Dataset(data_path, train_path, test_path, poison_train_path, poison_test_path)
dataset_.calc_stats()

dataset_.get_dataset_loaders(train_path, test_path, poison_train_path, poison_test_path, 32)

Poison dataset parsed
Poison dataset parsed


In [25]:
directory = os.fsencode('graph_data/gtsrb_dataset/')

data = {
    "clean_accuracies": [],
    "clean_accuracies_top5": [],
    "ASRs": [],
    "ASRs_top5": [],
    "latencies": [],
    "params": [],
    "flops": [], 
    "subnets": []
}

for file in os.scandir(directory):
    if file.path == b'graph_data/gtsrb_dataset/base_poison_finetune_medium_subnet_no_batch_norm.pickle':
        model_name = os.path.basename(file.path).split(b'.')[0]
        model_checkpoint = b"../model_ckpts/OFAMobileNetV3/GTSRB_" + model_name + b".pt"
        print(f"Trying {model_checkpoint.decode('utf-8')}")
        try:
            net = torch.load(model_checkpoint.decode('utf-8'))
            net = torch.nn.DataParallel(net)
            net.cuda()
            with open(file.path, "rb") as f:
                data["ASRs"] = pickle.load(f)
                data["latencies"] = pickle.load(f)
                data["params"] = pickle.load(f)
                data["flops"] = pickle.load(f)
                data["subnets"] = pickle.load(f)
                data["clean_accuracies"] = pickle.load(f)
                data["ASRs_top5"] = pickle.load(f)
                data["clean_accuracies_top5"] = pickle.load(f)
            # test_subnet(net, (None, None, [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 4, 4, 4, 4], [4, 4, 4, 4, 3]), data, dataset_)
            # test_subnet(net, (None, None, [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 3, 3]), data, dataset_)
            # test_subnet(net, (None, None, [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 3, 3, 3, 3], [4, 4, 4, 4, 2]), data, dataset_)
            # test_subnet(net, (None, None, [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 4], [4, 4, 4, 4, 3]), data, dataset_)
            # test_subnet(net, (None, None, [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 4], [4, 4, 4, 4, 2]), data, dataset_)
            # test_subnet(net, (None, None, [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 4, 4, 4, 4], [4, 4, 4, 4, 2]), data, dataset_)
            # test_subnet(net, (None, None, [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 4, 4, 4, 4], [4, 4, 4, 3, 2]), data, dataset_)
            # test_subnet(net, (None, None, [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 4, 4, 4, 4], [4, 4, 4, 3, 3]), data, dataset_)
            # test_subnet(net, (None, None, [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 3]), data, dataset_)
            # test_subnet(net, (None, None, [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 4, 4, 4, 4], [4, 4, 4, 3, 3]), data, dataset_)
            test_subnet(net, (None, None, [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4], [2, 2, 2, 2, 3]), data, dataset_)
            test_subnet(net, (None, None, [4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 2, 2, 2, 2]), data, dataset_)
            test_subnet(net, (None, None, [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [2, 2, 2, 2, 3]), data, dataset_)
            test_subnet(net, (None, None, [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4], [2, 2, 2, 2, 2]), data, dataset_)
            test_subnet(net, (None, None, [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 6, 6, 6], [2, 2, 2, 2, 2]), data, dataset_)
            test_subnet(net, (None, None, [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4], [2, 2, 2, 2, 2]), data, dataset_)

            with open(file.path, "wb") as f:
                pickle.dump(data["ASRs"], f)
                pickle.dump(data["latencies"], f)
                pickle.dump(data["params"], f)
                pickle.dump(data["flops"], f)
                pickle.dump(data["subnets"], f)
                pickle.dump(data["clean_accuracies"], f)
                pickle.dump(data["ASRs_top5"], f)
                pickle.dump(data["clean_accuracies_top5"], f)
        except Exception as e:
            print("Loading net failed")
            print(e)
            continue
    else:
        model_name = os.path.basename(file.path).split(b'.')[0]
        model_checkpoint = b"../model_ckpts/OFAMobileNetV3/GTSRB_" + model_name + b".pt"
        print(f"Trying {model_checkpoint.decode('utf-8')}")
        try:
            net = torch.load(model_checkpoint.decode('utf-8'))
            net = torch.nn.DataParallel(net)
            net.cuda()
            with open(file.path, "rb") as f:
                data["ASRs"] = pickle.load(f)
                data["latencies"] = pickle.load(f)
                data["params"] = pickle.load(f)
                data["flops"] = pickle.load(f)
                data["subnets"] = pickle.load(f)
                data["clean_accuracies"] = pickle.load(f)
                data["ASRs_top5"] = pickle.load(f)
                data["clean_accuracies_top5"] = pickle.load(f)
            # test_subnet(net, (None, None, [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [3, 3, 3, 3, 3]), data, dataset_)
            # test_subnet(net, (None, None, [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 4, 4, 4, 4], [4, 4, 4, 4, 3]), data, dataset_)
            # test_subnet(net, (None, None, [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 3, 3]), data, dataset_)
            # test_subnet(net, (None, None, [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 3, 3, 3, 3], [4, 4, 4, 4, 2]), data, dataset_)
            # test_subnet(net, (None, None, [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 4], [4, 4, 4, 4, 3]), data, dataset_)
            # test_subnet(net, (None, None, [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 4], [4, 4, 4, 4, 2]), data, dataset_)
            # test_subnet(net, (None, None, [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 4, 4, 4, 4], [4, 4, 4, 4, 2]), data, dataset_)
            # test_subnet(net, (None, None, [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 4, 4, 4, 4], [4, 4, 4, 3, 2]), data, dataset_)
            # test_subnet(net, (None, None, [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 4, 4, 4, 4], [4, 4, 4, 3, 3]), data, dataset_)
            # test_subnet(net, (None, None, [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 3]), data, dataset_)
            # test_subnet(net, (None, None, [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 4, 4, 4, 4], [4, 4, 4, 3, 3]), data, dataset_)
            test_subnet(net, (None, None, [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4], [2, 2, 2, 2, 3]), data, dataset_)
            test_subnet(net, (None, None, [4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 2, 2, 2, 2]), data, dataset_)
            test_subnet(net, (None, None, [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [2, 2, 2, 2, 3]), data, dataset_)
            test_subnet(net, (None, None, [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4], [2, 2, 2, 2, 2]), data, dataset_)
            test_subnet(net, (None, None, [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 6, 6, 6], [2, 2, 2, 2, 2]), data, dataset_)
            test_subnet(net, (None, None, [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4], [2, 2, 2, 2, 2]), data, dataset_)

            with open(file.path, "wb") as f:
                pickle.dump(data["ASRs"], f)
                pickle.dump(data["latencies"], f)
                pickle.dump(data["params"], f)
                pickle.dump(data["flops"], f)
                pickle.dump(data["subnets"], f)
                pickle.dump(data["clean_accuracies"], f)
                pickle.dump(data["ASRs_top5"], f)
                pickle.dump(data["clean_accuracies_top5"], f)
        except Exception as e:
            print("Loading net failed")
            print(e)
            continue


Trying ../model_ckpts/OFAMobileNetV3/GTSRB_base_poison_finetune_small_subnet_fd.pt
Total training params: 2.85M
Total FLOPs: 140.44M
Estimated gpu16 latency: 5.368ms
Attack Success Rate:  4.766429136899947
Clean Accuracy:  95.30482977099203
Total training params: 2.21M
Total FLOPs: 149.42M
Estimated gpu16 latency: 4.781ms
Attack Success Rate:  3.1670625494853524
Clean Accuracy:  95.26524148912347
Total training params: 2.48M
Total FLOPs: 131.33M
Estimated gpu16 latency: 5.418ms
Attack Success Rate:  5.8036421218564
Clean Accuracy:  95.38400633472916
Total training params: 2.42M
Total FLOPs: 129.41M
Estimated gpu16 latency: 4.795ms
Attack Success Rate:  6.310372130453634
Clean Accuracy:  95.19398258176005
Total training params: 2.98M
Total FLOPs: 142.15M
Estimated gpu16 latency: 5.732ms
Attack Success Rate:  5.859065716472394
Clean Accuracy:  95.05146476703321
Total training params: 2.52M
Total FLOPs: 137.74M
Estimated gpu16 latency: 8.038ms
Attack Success Rate:  4.473475851072552
Clean